# TVTSyn inference

First run `bash inference/setup_listening.sh` from the repository root. In VS Code, select the interpreter `.venv-listening/bin/python` for this notebook (TVTSyn listening, CPU). Restart the kernel if you changed environments. This environment supports all three modes and target-speaker extraction.

The first cell verifies the selected interpreter. The first run downloads the pinned public model from Hugging Face. Set `MODEL` to a local bundle for offline use. These examples run on CPU and do not need a working GPU. The separate listening review is documented in `docs/LISTENING_VALIDATION.md`.

Notebook audio players normalize playback individually. Use the listening review page for comparisons with shared gain.


In [ ]:
import sys, importlib.util
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
print("Notebook interpreter:", sys.executable)
if importlib.util.find_spec("tvtsyn_inference") is None:
    raise RuntimeError("Select the repository .venv-listening/bin/python kernel after running bash inference/setup_listening.sh. Current interpreter: " + sys.executable)


In [ ]:
from pathlib import Path
import numpy as np
from IPython.display import Audio, display
from tvtsyn_inference import TVTSyn

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
MODEL = 'warisqr007/TVTSyn'  # Or str(ROOT / 'artifacts/TVTSyn')
REVISION = '2d3147b8a465e1d9aad33a2a0e53956bf8ed32aa'
source = ROOT / 'samples/source.wav'
target = ROOT / 'samples/target.wav'

## PyTorch whole utterance and target embedding

In [ ]:
pt = TVTSyn.from_pretrained(MODEL, backend='pytorch', revision=REVISION)
speaker = pt.encode_speaker(target)
output = ROOT / 'outputs'
output.mkdir(exist_ok=True)
np.save(output / 'target.npy', speaker)
display(Audio(pt.convert_file(source, speaker), rate=16000))

## PyTorch streaming

In [ ]:
display(Audio(pt.convert_file(source, speaker, mode='streaming'), rate=16000))

## ONNX streaming

With a saved target embedding, this backend does not require PyTorch or SpeechBrain.

In [ ]:
onnx = TVTSyn.from_pretrained(MODEL, backend='onnx', revision=REVISION)
display(Audio(onnx.convert_file(source, output / 'target.npy'), rate=16000))

## Incremental input and final flush

In [ ]:
import soundfile as sf
x, sr = sf.read(source, dtype='float32')
stream = onnx.stream(output / 'target.npy', sample_rate=sr)
try:
    chunks = [stream.push(x[i:i+731]) for i in range(0, len(x), 731)]
    chunks.append(stream.flush())
finally:
    stream.close()
display(Audio(np.concatenate(chunks), rate=16000))